# Understand the Collab API flow

Learn how a server workflow calls published client methods, receives their results, and decides what happens next. By the end, you should be able to trace one round through the code and write a small coordinating workflow of your own.

This lesson uses a sample-weighted NumPy linear-regression example. Each site generates its own synthetic training data. There is no dataset download or GPU requirement.

The main example uses class methods in separate server and client files. Collab also supports module-level functions, shared files, and mixed organizations. Those choices change how you package the code; the call-and-return flow is the same.

## Start with the execution flow

![Collab execution flow: job.py starts the collaboration, the server calls both clients, clients return local results, and the server aggregates them before the next round.](collab_flow.svg)

1. **Configure the application in `job.py`.** Construct `FedAvg()` and `Trainer()`, place them in `CollabRecipe`, and set each site's seed and sample count. These statements describe the application; they do not run a training round.
2. **Start it with `recipe.execute(SimEnv(...))`.** NVFLARE starts the server and clients and makes their published methods available for calls. The server's `@collab.main` method is the workflow entry point.
3. **The server owns the round loop.** `FedAvg.run()` initializes zero weights and starts a round by calling `collab.clients(timeout=60).train(weights)`.
4. **Each client runs its own published method.** The runtime dispatches that call to `Trainer.train(weights)` at each site. Each invocation receives the global weights and uses that site's own configuration and training rows.
5. **Clients return local results.** Each method returns trained weights, its sample count, and post-training local loss. The blocking group call waits for the client outcomes and returns them to the server.
6. **The server checks outcomes and aggregates.** It checks for failures, computes sample-weighted averages, and uses the new global weights in the next round. After three rounds it saves the final weights and round history, then returns from `run()`.

`@collab.main` is invoked once for this job. The Python loop inside it creates the three rounds. Each client's `train()` is invoked once per round and returns once per call; the client has no separate receive/send loop in this example.

## Map the files to responsibilities

| File | Runs where? | Owns what? |
| --- | --- | --- |
| `job.py` | Where you submit the application | Recipe, participating sites, per-site configuration, and execution environment |
| `server.py` | Server | Round loop, global weights, client calls, failure policy, aggregation, and saved artifacts |
| `client.py` | Each participating client | Published `train()` method, local data generation, and local training |
| `model.py` | Imported at each client | Framework-independent gradient descent |

The runtime transports calls and results. The averaging rule, sample weighting, number of rounds, and persistence are choices made by this application's server code.

## Choose the code organization

The decorators identify roles and callable entry points. Separate files and classes are optional.

| Organization | Where the decorated code lives | What to pass to `CollabRecipe` |
| --- | --- | --- |
| Class methods, separate files | `FedAvg.run()` in `server.py`; `Trainer.train()` in `client.py` | `server=FedAvg(), client=Trainer()` — the main example below |
| Class methods, one file | Both classes in a shared module or the launcher script | Instances of those classes: `server=FedAvg(), client=Trainer()` |
| Module-level functions, separate files | `@collab.main def run()` in a server module; `@collab.publish def train(...)` in a client module | The imported module objects as `server` and `client` |
| Module-level functions, one file | Both decorated functions in the same module | That module for both roles, or omit both arguments when the recipe is constructed in that same module |

Class instances and modules can also be mixed between roles. For example, a class-based server can coordinate a client defined by module-level functions.

### Module-level functions in a shared file

Here is a minimal call-and-return example illustrating organization. It asks sites to identify themselves; the training example later in the notebook adds the round loop and aggregation.

```python
# collaboration.py
from nvflare.collab import collab

@collab.publish
def identify():
    return collab.site_name

@collab.main
def run():
    responses = collab.clients.identify()
    if responses.failures:
        raise RuntimeError(f"Client calls failed: {responses.failures}")
    print(dict(responses))
```

The launcher supplies the same module for both roles:

```python
# job.py
import collaboration
from nvflare.collab import CollabRecipe
from nvflare.recipe import SimEnv

if __name__ == "__main__":
    recipe = CollabRecipe(
        job_name="identify_sites",
        server=collaboration,
        client=collaboration,
        min_clients=2,
    )
    recipe.execute(SimEnv(num_clients=2))
```

If the decorated functions and launcher all live in the same `job.py`, this constructor uses that script's module automatically:

```python
recipe = CollabRecipe(job_name="identify_sites", min_clients=2)
```

Keep execution under `if __name__ == "__main__":` so importing the module at a remote site does not launch another simulation.

### Module-level functions in separate files

If you place the decorated `run()` and client functions at module level in separate files, pass those modules:

```python
import server as server_module
import client as client_module

recipe = CollabRecipe(
    job_name="my_collaboration",
    server=server_module,
    client=client_module,
    min_clients=2,
)
```

This constructor assumes those modules contain the decorated functions. The main training example instead imports the classes from its files and supplies instances. In either organization, keep distributed application code in importable Python files or packages included in the job; use the notebook to inspect and launch that code. Local helper functions only need publishing if they are intended as remote entry points.

## Install lesson dependencies

First complete the [course environment setup](../02.0_introduction/introduction.ipynb) and select its kernel. The requirements below add only the dependencies needed by this lesson.

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import Code, display

# Make notebook shell commands use the selected kernel's environment.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")

lesson = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "collab_api.ipynb").is_file())
code_dir = lesson / "code"
print("Example code:", code_dir)
%cd {code_dir}

In [ ]:
%cd {code_dir}
!python -m pip install -r requirements.txt
assert _exit_code == 0, "Command failed; see the output above."

## Understand the two decorators

`@collab.main` identifies the server's workflow entry point. In this example, it decorates `FedAvg.run()`.

`@collab.publish` makes a method available to remote calls. It decorates `Trainer.train(weights)` at the clients. The server calls that method through a Collab proxy:

```python
responses = collab.clients(timeout=60).train(weights)
```

Read this line from left to right:

| Part | Meaning |
| --- | --- |
| `collab.clients` | The participating clients available in the running collaboration |
| `(timeout=60)` | Call options; this call expects results and blocks by default |
| `.train(weights)` | Invoke the published `train` method at every client, passing the same global weights |
| `responses` | Successful site/result pairs and a separate mapping of failures |

The client calls are dispatched in parallel. Blocking means the server waits at this line before aggregating; it does not mean the clients train one after another. `collab.clients` needs an active Collab runtime context, so entering this call in an arbitrary notebook cell does not start a collaboration.

The method name and arguments must match the published client interface. Local helper functions such as `model.train()` do not need to be published because the client calls those helpers locally.

## Read the client input and result contract

The server passes a three-element NumPy weight vector. The published method returns a dictionary:

| Field | Type | Interpretation |
| --- | --- | --- |
| `weights` | NumPy array, shape `(3,)` | Locally trained model, starting from the received global weights |
| `samples` | Integer | Number of local training rows; used by this server as the aggregation weight |
| `loss` | Float | MSE on that site's training data after local training |

Collab transports the arguments and returned values. Your application defines this dictionary's keys and meanings; there is no `FLModel` envelope in this example. Both sites must return compatible vector shapes and fields for the aggregation code to work.

The training helper copies the received weights before updating them. Each client regenerates the same deterministic site-local dataset on each call because its seed and sample count stay fixed. A larger application can load and retain local data once for the job rather than regenerate it.

Raw training rows stay at the site. Model updates and returned statistics can still disclose information; this example adds no privacy mechanism.

In [ ]:
display(Code(filename=str(code_dir / "model.py"), language="python"))
display(Code(filename=str(code_dir / "client.py"), language="python"))

## Follow one round through the server

At the start of a round, the server has one global weight vector. Both clients start from that same vector, then train on their own rows.

The per-site configuration gives `site-1` 80 samples and `site-2` 120 samples. After both replies arrive, the server computes:

```text
next global weights = (80 × site-1 weights + 120 × site-2 weights) / 200
                    = 0.4 × site-1 weights + 0.6 × site-2 weights
```

The next iteration sends **this aggregated vector** to both sites. It does not send each site back its own previous local model. `samples` has this effect because the server explicitly uses it in the formula; a dictionary field does not automatically select an aggregation policy.

The same weighting is applied to the reported local losses. That number is a weighted mean of **post-training local losses**, computed on training data at the two sites. It is not a validation metric for the aggregated global model. To measure global-model quality, add a distinct evaluation step with a clearly defined dataset and metric.

Read the following server code in order: initialize → call clients → collect outcomes → check failures → aggregate → record progress → repeat → save. `log_progress` marks the application's own round messages for progress logging mode.

In [ ]:
display(Code(filename=str(code_dir / "server.py"), language="python"))

## Handle successful replies and failed calls separately

For a completed blocking group call:

```python
results = dict(responses)
```

creates a mapping from site name to that site's returned dictionary. This mapping contains successful replies. `responses.failures` maps failed sites to their call errors; timeouts and remote exceptions need attention even when another site returned successfully.

This server stops the job if any client failed or if there are no successful replies. It checks that condition before aggregation, so a missing site is not silently treated as a zero-valued update. The example does not implement retry or partial-participation policies. Invalid method names or arguments can also fail before dispatch; inspect the exception and verify the published interface.

The call's `timeout=60` bounds waiting for client results. It does not change the number of rounds or guarantee that a remote computation has been cancelled. Local training that can run for a long time should cooperate with job aborts, for example by checking `collab.is_aborted` at appropriate boundaries.

## Configure and launch the collaboration

`CollabRecipe(server=FedAvg(), client=Trainer(), ...)` packages these objects for their respective roles. `recipe.set_per_site_config(...)` makes each site's values available through `collab.get_app_prop(...)` inside its method calls.

| Site | Seed | Samples | Aggregation fraction |
| --- | --- | --- | --- |
| `site-1` | 1 | 80 | 0.4 |
| `site-2` | 2 | 120 | 0.6 |

`min_clients=2` requires two clients to start this job. The separate `responses.failures` check defines what this server does if a later method call fails.

`add_client_file("model.py")` includes the local training helper with the exported client code. Exporting the recipe packages code and configuration; it does not call the decorated training methods.

In [ ]:
display(Code(filename=str(code_dir / "job.py"), language="python"))

## Run and observe the flow

`SimEnv(log_config="progress")` shows the server's three round messages and final run summary. Expect one history entry per round. Detailed server and client logs remain in the workspace.

The notebook uses the same command pattern as the Client API lesson: select the course kernel, enter the local `code/` directory, and run `!python job.py`. A failed command stops execution before later cells read old artifacts.

In [ ]:
%cd {code_dir}
!python job.py
assert _exit_code == 0, "Command failed; see the output above."

## Inspect the saved artifacts

When `FedAvg.run()` finishes, it explicitly writes `weights.npy` and `history.json` in the server's run directory. Returning weights from `run()` alone is not the persistence mechanism used here.

The weights contain the final global vector. History contains the sample-weighted local training loss for each round; read it using the metric definition above.

In [ ]:
import json
import numpy as np

result = Path("/tmp/nvflare/chapter2/collab/collab_fedavg/server/simulate_job")
history = json.loads((result / "history.json").read_text())
for row in history:
    print(f"Round {row['round']}: weighted local training loss = {row['local_loss']:.4f}")
print("Final global weights:", np.load(result / "weights.npy"))
print("Server log:", result.parents[1] / "server" / "log.txt")

## Export the same application

The following command exports the recipe without running the round loop. It uses the same server, client, per-site properties, and helper file. Submit the exported job to an appropriately configured deployment when moving beyond local simulation.

In [ ]:
%cd {code_dir}
!python job.py --export --export-dir /tmp/nvflare/chapter2/exports/collab
assert _exit_code == 0, "Command failed; see the output above."

## Connect this flow to the Client API lesson

| Responsibility | Client API example | Collab API example |
| --- | --- | --- |
| Start the application | Execute a built-in NumPy FedAvg recipe | Execute `CollabRecipe` with a server and client |
| Decide when clients compute | Built-in server workflow assigns tasks | Your `@collab.main` workflow calls published methods |
| Run local training | Client receives a model, computes, and sends a result | Published client method receives arguments, computes, and returns a result |
| Describe results | `FLModel` fields interpreted by the workflow | Application-defined method arguments and return values |
| Choose aggregation and weights | Configure the built-in recipe's aggregation contract | Implement the formula in server code |
| End a round | Return task results to the workflow | Collect method-call outcomes and continue the server loop |

Both APIs use site-local computation and recipes for execution and export. Choose the approach that fits who should own the coordination logic. The two lessons use independently configured NumPy examples; their numeric results reflect their own datasets and training settings.

## Trace a change before running it

- Change `server=FedAvg()` to `server=FedAvg(num_rounds=1)` in `job.py`. Predict how many times each client's published method will run and how many history entries will be saved.
- Change the second site's sample count from 120 to 80. What aggregation fractions should the server now use?
- Explain why each next-round call passes the aggregated global weights even though the same client application remains available for the job.
- Identify the exact code you would change to report validation of the aggregated model instead of post-training local loss.

Use the saved history and server log to check the control flow after a change.

## Troubleshoot the flow

| Symptom | What to inspect |
| --- | --- |
| The server cannot call `train` | The client decorator, published method name, and argument signature |
| The group call waits | Client logs for long-running training or an exception; distinguish startup from a method-call timeout |
| Some results are missing | `responses.failures`; successful replies alone are not evidence that all clients completed |
| Aggregation raises a key or shape error | The returned dictionary schema and the dimensions of `weights` at each site |
| Changing sample counts has no expected effect | The per-site property read in the client and the server's explicit weighting formula |
| A lower displayed loss is interpreted as better global validation | The metric is post-training local loss; add a separate global-model evaluation |
| A helper import fails in an exported job | Confirm its source file is included, as `model.py` is through `add_client_file` |

The basic flow above is blocking and server-coordinated. Once it is clear, continue to the optional features below.

## Extend the call pattern

To call selected sites, use `collab.get_clients(["site-1"])`. For a nonblocking group call, place `blocking=False` before the method call, consume the returned result stream, and inspect failures after the stream completes. The server then owns the scheduling and aggregation policy for those arriving results.

See the [advanced Collab examples](https://github.com/NVIDIA/NVFlare/tree/main/examples/advanced/collab) for callbacks, split learning, and decentralized coordination.

Continue with the [recap](../02.5_recap/recap.ipynb).